# 05. Quarterly flows and predictors

Convert year-to-date figures into quarterly flows and construct bank predictors. Missing values are retained at this stage. Notebook 09 applies the exact previous-quarter match.


## 1. Mount Drive, import configuration and validate the Notebook 04 handoff


In [ ]:
# Mount Drive, import main analysis configuration and validate the Notebook 04 handoff

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import importlib.util
import json
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display

try:
    import pyarrow  # noqa: F401
except ImportError:
    print("Installing the missing pyarrow Parquet dependency...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow"], check=True)
    import pyarrow  # noqa: F401

PROJECT_ROOT = Path("/content/drive/MyDrive/Data Final")
CONFIG_PATH = PROJECT_ROOT / "Version 2 Main Analysis" / "config.py"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Shared V2 config not found: {CONFIG_PATH}")

spec = importlib.util.spec_from_file_location("v2_config", CONFIG_PATH)
config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(config)

assert config.ANALYSIS_VERSION == "v2_bank_lag1_main"
assert config.BANK_PREDICTOR_SOURCE == "exact_t_minus_1"
assert config.BANK_PREDICTOR_LAG_QUARTERS == 1
assert config.ARCHIVED_V1_READ_ONLY is True

V2_OUTPUT_ROOT = config.V2_OUTPUT_ROOT
V2_PROCESSED_DATA_DIR = config.V2_PROCESSED_DATA_DIR
NB04_OUTPUT_DIR = V2_OUTPUT_ROOT / "04_Commercial_Bank_Filter"
NB05_OUTPUT_DIR = V2_OUTPUT_ROOT / "05_Quarterly_Flows_and_Candidate_Predictors"
INPUT_PATH = V2_PROCESSED_DATA_DIR / "commercial_bank_ffiec_panel_v2.parquet"
OUTPUT_PATH = V2_PROCESSED_DATA_DIR / "bank_predictor_source_v2.parquet"
NB04_AUDIT_PATH = NB04_OUTPUT_DIR / "audit_report_v2.json"
NB05_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [INPUT_PATH, NB04_AUDIT_PATH]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError("Required Notebook 04 handoff missing: " + "; ".join(missing_paths))

with NB04_AUDIT_PATH.open("r", encoding="utf-8") as file:
    nb04_audit = json.load(file)
if nb04_audit.get("overall_status") != "PASS":
    raise AssertionError("Notebook 04 did not report PASS.")

panel = pd.read_parquet(INPUT_PATH)
required_columns = [
    "report_date", "idrssd", "cert", "bank_name", "filing_form", "institution_class",
    "total_assets", "total_loans", "total_deposits", "total_equity_capital",
    "past_due_90_days", "nonaccrual_loans", "allowance_credit_losses",
    "other_real_estate_owned", "available_for_sale_securities",
    "held_to_maturity_securities", "accumulated_other_comprehensive_income",
    "trading_assets", "net_income_ytd", "net_interest_income_ytd",
    "provision_credit_losses_ytd", "total_interest_expense_ytd",
    "total_interest_income_ytd", "total_noninterest_expense_ytd",
    "total_noninterest_income_ytd",
]
missing_columns = [column for column in required_columns if column not in panel.columns]
if missing_columns:
    raise KeyError(f"Notebook 04 panel is missing required fields: {missing_columns}")

panel["report_date"] = pd.to_datetime(panel["report_date"], errors="raise").dt.normalize()
panel["idrssd"] = pd.to_numeric(panel["idrssd"], errors="raise").astype("int64")
panel["cert"] = pd.to_numeric(panel["cert"], errors="raise").astype("int64")
panel = panel.sort_values(["idrssd", "report_date"], kind="mergesort").reset_index(drop=True)

duplicate_input_keys = panel.loc[panel.duplicated(["idrssd", "report_date"], keep=False)]
expected_quarters = pd.period_range("2001Q1", "2026Q1", freq="Q").to_timestamp(how="end").normalize()
observed_quarters = pd.DatetimeIndex(sorted(panel["report_date"].unique()))

input_gate = pd.DataFrame({
    "check": [
        "Notebook 04 status is PASS", "Input is nonempty", "Required fields exist",
        "Only commercial classes N, NM and SM remain", "IDRSSD-source quarter is unique",
        "Complete 2001Q1-2026Q1 reporting grid", "Total assets is positive",
        "No target fields are present", "No lag1 fields are present",
    ],
    "passed": [
        nb04_audit.get("overall_status") == "PASS", len(panel) > 0, not missing_columns,
        set(panel["institution_class"].dropna()).issubset({"N", "NM", "SM"}),
        duplicate_input_keys.empty, observed_quarters.equals(expected_quarters),
        panel["total_assets"].gt(0).all(),
        not any(column in panel.columns for column in ["failed_1yr", "failure_date", "days_to_failure"]),
        not any(column.endswith("_lag1") for column in panel.columns),
    ],
})
input_gate["status"] = input_gate["passed"].map({True: "PASS", False: "FAIL"})
display(input_gate)
if not input_gate["passed"].all():
    raise AssertionError("Notebook 05 input gate failed.")

print(f"Input observations: {len(panel):,}")
print(f"Unique IDRSSD: {panel['idrssd'].nunique():,}")
print(f"Reporting quarters: {panel['report_date'].nunique():,}")
print(f"Period: {panel['report_date'].min().date()} to {panel['report_date'].max().date()}")


## 2. Lock the earlier feature manifest and the analysis source-quarter contract


In [ ]:
# Lock the earlier analysis feature manifest and the main analysis source-quarter contract

# These names reproduce the locked earlier analysis bank specification exactly:
# 18 selected bank predictors plus the history indicator = 19 bank features.
LOCKED_V1_BANK_PREDICTORS = [
    "log_total_assets",
    "equity_to_assets",
    "noncurrent_loans_to_loans",
    "allowance_to_loans",
    "provisions_to_average_assets",
    "other_real_estate_owned_to_assets",
    "return_on_average_assets",
    "net_interest_margin",
    "interest_expense_to_average_assets",
    "noninterest_income_to_average_assets",
    "efficiency_ratio",
    "loans_to_assets",
    "deposits_to_assets",
    "securities_to_assets",
    "aoci_to_assets",
    "annual_asset_growth",
    "annual_loan_growth",
    "annual_deposit_growth",
]
HISTORY_INDICATOR = "insufficient_history_5q"
LOCKED_V1_BANK_FEATURES = LOCKED_V1_BANK_PREDICTORS + [HISTORY_INDICATOR]
AUDITED_CANDIDATE_ONLY = ["loans_to_deposits"]

if len(LOCKED_V1_BANK_PREDICTORS) != 18 or len(LOCKED_V1_BANK_FEATURES) != 19:
    raise AssertionError("Locked Version 1 bank manifest must contain 18 predictors plus one history indicator.")
if len(set(LOCKED_V1_BANK_FEATURES)) != len(LOCKED_V1_BANK_FEATURES):
    raise AssertionError("Locked Version 1 feature names are not unique.")

locked_feature_manifest = pd.DataFrame({
    "feature_order": range(1, 20),
    "feature": LOCKED_V1_BANK_FEATURES,
    "version_1_status": ["Selected bank predictor"] * 18 + ["Locked history indicator"],
    "version_2_source_name": LOCKED_V1_BANK_FEATURES,
    "notebook_09_name": [f"{feature}_lag1" for feature in LOCKED_V1_BANK_FEATURES],
    "timing_in_notebook_05": "Contemporaneous at source quarter s; information through s only",
})

source_contract = pd.DataFrame({
    "rule": [
        "Source key", "Bank history direction", "Predictor shifting", "Target attachment",
        "Notebook 09 match", "Notebook 09 rename", "Earliest source quarter",
        "First eligible V2 reference quarter", "Imputation", "Global winsorisation",
    ],
    "locked_value": [
        "IDRSSD + source_report_date", "Backward-looking through s only", "None",
        "None", "s = t-1 exact calendar quarter", "Every bank feature receives _lag1",
        "2001Q1", "2001Q2", "None", "None",
    ],
})
display(locked_feature_manifest)
display(source_contract)


## 3. Convert RIAD YTD fields into true quarterly flows


In [ ]:
# Convert RIAD YTD fields into true quarterly flows

YTD_FLOW_MAP = {
    "net_income_ytd": "net_income_quarterly",
    "net_interest_income_ytd": "net_interest_income_quarterly",
    "provision_credit_losses_ytd": "provision_credit_losses_quarterly",
    "total_interest_expense_ytd": "total_interest_expense_quarterly",
    "total_interest_income_ytd": "total_interest_income_quarterly",
    "total_noninterest_expense_ytd": "total_noninterest_expense_quarterly",
    "total_noninterest_income_ytd": "total_noninterest_income_quarterly",
}

grouped_dates = panel.groupby("idrssd", sort=False)["report_date"]
previous_report_date = grouped_dates.shift(1)
current_qindex = panel["report_date"].dt.year * 4 + panel["report_date"].dt.quarter
previous_qindex = previous_report_date.dt.year * 4 + previous_report_date.dt.quarter
exact_previous_calendar_quarter = current_qindex.sub(previous_qindex).eq(1)
same_calendar_year = previous_report_date.dt.year.eq(panel["report_date"].dt.year)
is_q1 = panel["report_date"].dt.quarter.eq(1)

quarterly_flow_audit_records = []
for ytd_column, quarterly_column in YTD_FLOW_MAP.items():
    current_ytd = pd.to_numeric(panel[ytd_column], errors="coerce").astype("float64")
    previous_ytd = current_ytd.groupby(panel["idrssd"], sort=False).shift(1)
    quarterly = pd.Series(np.nan, index=panel.index, dtype="float64")

    q1_eligible = is_q1 & current_ytd.notna()
    subtraction_eligible = (
        ~is_q1 & exact_previous_calendar_quarter & same_calendar_year
        & current_ytd.notna() & previous_ytd.notna()
    )
    quarterly.loc[q1_eligible] = current_ytd.loc[q1_eligible]
    quarterly.loc[subtraction_eligible] = current_ytd.loc[subtraction_eligible] - previous_ytd.loc[subtraction_eligible]
    panel[quarterly_column] = quarterly

    q1_mismatches = int((~np.isclose(
        panel.loc[q1_eligible, quarterly_column], current_ytd.loc[q1_eligible],
        rtol=0, atol=1e-12, equal_nan=False,
    )).sum())
    subtraction_expected = current_ytd.loc[subtraction_eligible] - previous_ytd.loc[subtraction_eligible]
    subtraction_mismatches = int((~np.isclose(
        panel.loc[subtraction_eligible, quarterly_column], subtraction_expected,
        rtol=0, atol=1e-12, equal_nan=False,
    )).sum())
    unavailable_non_q1 = ~is_q1 & ~subtraction_eligible
    quarterly_flow_audit_records.append({
        "ytd_column": ytd_column, "quarterly_column": quarterly_column,
        "q1_values_copied": int(q1_eligible.sum()), "q1_formula_mismatches": q1_mismatches,
        "q2_q4_values_differenced": int(subtraction_eligible.sum()),
        "q2_q4_formula_mismatches": subtraction_mismatches,
        "q2_q4_missing_no_exact_predecessor_or_value": int(unavailable_non_q1.sum()),
        "negative_quarterly_values_retained": int(quarterly.lt(0).sum()),
        "missing_quarterly_values": int(quarterly.isna().sum()),
    })

quarterly_flow_audit = pd.DataFrame(quarterly_flow_audit_records)
quarterly_flow_columns = list(YTD_FLOW_MAP.values())
flow_gate = pd.DataFrame({
    "check": [
        "Seven YTD fields converted", "All Q1 formulas match", "All Q2-Q4 formulas match",
        "No non-Q1 value created without exact predecessor", "No infinite quarterly flows",
    ],
    "passed": [
        len(quarterly_flow_columns) == 7,
        quarterly_flow_audit["q1_formula_mismatches"].eq(0).all(),
        quarterly_flow_audit["q2_q4_formula_mismatches"].eq(0).all(),
        all(panel.loc[~is_q1 & ~(
            exact_previous_calendar_quarter & same_calendar_year
            & panel[ytd].notna()
            & panel[ytd].groupby(panel["idrssd"], sort=False).shift(1).notna()
        ), quarterly].isna().all() for ytd, quarterly in YTD_FLOW_MAP.items()),
        np.isfinite(panel[quarterly_flow_columns].to_numpy(dtype="float64"), where=panel[quarterly_flow_columns].notna().to_numpy(), out=np.ones(panel[quarterly_flow_columns].shape, dtype=bool)).all(),
    ],
})
flow_gate["status"] = flow_gate["passed"].map({True: "PASS", False: "FAIL"})
display(flow_gate)
display(quarterly_flow_audit)
if not flow_gate["passed"].all():
    raise AssertionError("Quarterly-flow conversion gate failed.")


## 4. Construct backward-looking history, TTM flows and strict average denominators


In [ ]:
# Construct backward-looking history, TTM flows and strict average denominators

# A new reporting spell begins whenever the immediately preceding calendar quarter is absent.
gap_or_first = previous_report_date.isna() | ~exact_previous_calendar_quarter
reporting_spell = gap_or_first.groupby(panel["idrssd"], sort=False).cumsum()
panel["consecutive_quarters_observed"] = (
    panel.groupby([panel["idrssd"], reporting_spell], sort=False).cumcount() + 1
).astype("int64")
panel[HISTORY_INDICATOR] = panel["consecutive_quarters_observed"].lt(5).astype("int8")

TTM_FLOW_MAP = {
    quarterly_column: quarterly_column.replace("_quarterly", "_ttm")
    for quarterly_column in quarterly_flow_columns
}
ttm_flow_audit_records = []
for quarterly_column, ttm_column in TTM_FLOW_MAP.items():
    grouped_flow = panel.groupby("idrssd", sort=False)[quarterly_column]
    components = [grouped_flow.shift(lag) for lag in range(4)]
    complete_components = pd.concat(components, axis=1).notna().all(axis=1)
    ttm_eligible = panel["consecutive_quarters_observed"].ge(4) & complete_components
    expected = pd.concat(components, axis=1).sum(axis=1, min_count=4)
    constructed = expected.where(ttm_eligible)
    panel[ttm_column] = constructed.astype("float64")
    formula_match = np.isclose(
        panel.loc[ttm_eligible, ttm_column], expected.loc[ttm_eligible],
        rtol=0, atol=1e-12, equal_nan=False,
    )
    ttm_flow_audit_records.append({
        "quarterly_column": quarterly_column, "ttm_column": ttm_column,
        "eligible_complete_windows": int(ttm_eligible.sum()),
        "formula_mismatches": int((~formula_match).sum()),
        "missing_ttm_values": int(constructed.isna().sum()),
        "negative_ttm_values_retained": int(constructed.lt(0).sum()),
    })

ttm_flow_audit = pd.DataFrame(ttm_flow_audit_records)
ttm_flow_columns = list(TTM_FLOW_MAP.values())

BALANCE_COLUMNS = ["total_assets", "total_loans", "total_deposits", "total_equity_capital"]
average_columns, lag4_columns, average_audit_records = [], [], []
for balance_column in BALANCE_COLUMNS:
    grouped_balance = panel.groupby("idrssd", sort=False)[balance_column]
    components = [grouped_balance.shift(lag) for lag in range(5)]
    component_frame = pd.concat(components, axis=1)
    complete = component_frame.notna().all(axis=1)
    eligible = panel["consecutive_quarters_observed"].ge(5) & complete
    average_column = f"average_{balance_column}_5q_strict"
    lag4_column = f"{balance_column}_lag4_strict"
    panel[average_column] = component_frame.mean(axis=1).where(eligible).astype("float64")
    panel[lag4_column] = components[4].where(eligible).astype("float64")
    average_columns.append(average_column)
    lag4_columns.append(lag4_column)
    average_audit_records.append({
        "balance_column": balance_column, "average_column": average_column,
        "lag4_column": lag4_column, "eligible_five_quarter_windows": int(eligible.sum()),
        "missing_average": int(panel[average_column].isna().sum()),
        "missing_lag4": int(panel[lag4_column].isna().sum()),
    })
average_denominator_audit = pd.DataFrame(average_audit_records)

# Temporal controls: everything above is calculated from current and earlier rows only.
history_gate = pd.DataFrame({
    "check": [
        "Consecutive history is at least one", "First row of every spell has history one",
        "History indicator equals count below five", "Seven TTM flows constructed",
        "All TTM formulas match", "Four strict five-quarter averages constructed",
        "Four strict lag-four fields constructed",
    ],
    "passed": [
        panel["consecutive_quarters_observed"].ge(1).all(),
        panel.loc[gap_or_first, "consecutive_quarters_observed"].eq(1).all(),
        panel[HISTORY_INDICATOR].eq(panel["consecutive_quarters_observed"].lt(5).astype("int8")).all(),
        len(ttm_flow_columns) == 7, ttm_flow_audit["formula_mismatches"].eq(0).all(),
        len(average_columns) == 4, len(lag4_columns) == 4,
    ],
})
history_gate["status"] = history_gate["passed"].map({True: "PASS", False: "FAIL"})
display(history_gate)
display(ttm_flow_audit)
display(average_denominator_audit)
if not history_gate["passed"].all():
    raise AssertionError("History and trailing-measure gate failed.")


## 5. Construct the CAMELS candidate predictor library


In [ ]:
# Construct the CAMELS candidate predictor library

def safe_ratio(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce").astype("float64")
    denominator = pd.to_numeric(denominator, errors="coerce").astype("float64")
    eligible = numerator.notna() & denominator.notna() & denominator.gt(0)
    result = pd.Series(np.nan, index=panel.index, dtype="float64")
    result.loc[eligible] = numerator.loc[eligible] / denominator.loc[eligible]
    return result

average_assets = panel["average_total_assets_5q_strict"]
operating_revenue_ttm = panel["net_interest_income_ttm"] + panel["total_noninterest_income_ttm"]

panel["log_total_assets"] = np.log(panel["total_assets"].where(panel["total_assets"].gt(0)))
panel["equity_to_assets"] = safe_ratio(panel["total_equity_capital"], panel["total_assets"])
panel["noncurrent_loans_to_loans"] = safe_ratio(
    panel["past_due_90_days"] + panel["nonaccrual_loans"], panel["total_loans"]
)
panel["allowance_to_loans"] = safe_ratio(panel["allowance_credit_losses"], panel["total_loans"])
panel["provisions_to_average_assets"] = safe_ratio(panel["provision_credit_losses_ttm"], average_assets)
panel["other_real_estate_owned_to_assets"] = safe_ratio(panel["other_real_estate_owned"], panel["total_assets"])
panel["return_on_average_assets"] = safe_ratio(panel["net_income_ttm"], average_assets)
panel["net_interest_margin"] = safe_ratio(panel["net_interest_income_ttm"], average_assets)
panel["interest_expense_to_average_assets"] = safe_ratio(panel["total_interest_expense_ttm"], average_assets)
panel["noninterest_income_to_average_assets"] = safe_ratio(panel["total_noninterest_income_ttm"], average_assets)
panel["efficiency_ratio"] = safe_ratio(panel["total_noninterest_expense_ttm"], operating_revenue_ttm)
panel["loans_to_assets"] = safe_ratio(panel["total_loans"], panel["total_assets"])
panel["loans_to_deposits"] = safe_ratio(panel["total_loans"], panel["total_deposits"])
panel["deposits_to_assets"] = safe_ratio(panel["total_deposits"], panel["total_assets"])
panel["securities_to_assets"] = safe_ratio(
    panel["available_for_sale_securities"] + panel["held_to_maturity_securities"], panel["total_assets"]
)

# B530 AOCI is retained as the historically consistent signed sensitivity proxy.
# It is not claimed to be a pure securities-only unrealised-gain measure.
panel["unrealised_securities_gains_losses_proxy"] = panel["accumulated_other_comprehensive_income"].astype("float64")
panel["aoci_to_assets"] = safe_ratio(
    panel["unrealised_securities_gains_losses_proxy"], panel["total_assets"]
)

for current_column, predictor in [
    ("total_assets", "annual_asset_growth"),
    ("total_loans", "annual_loan_growth"),
    ("total_deposits", "annual_deposit_growth"),
]:
    lag4 = panel[f"{current_column}_lag4_strict"]
    panel[predictor] = safe_ratio(panel[current_column], lag4) - 1.0
    panel.loc[lag4.isna() | lag4.le(0), predictor] = np.nan

candidate_predictors = LOCKED_V1_BANK_PREDICTORS + AUDITED_CANDIDATE_ONLY
predictor_definitions = [
    ("log_total_assets", "Size", "ln(total_assets)", "total_assets > 0"),
    ("equity_to_assets", "Capital", "total_equity_capital / total_assets", "total_assets > 0"),
    ("noncurrent_loans_to_loans", "Asset quality", "(past_due_90_days + nonaccrual_loans) / total_loans", "total_loans > 0"),
    ("allowance_to_loans", "Asset quality", "allowance_credit_losses / total_loans", "total_loans > 0; signed CECL values retained"),
    ("provisions_to_average_assets", "Asset quality", "provision_credit_losses_ttm / average_total_assets_5q_strict", "five-quarter average assets > 0"),
    ("other_real_estate_owned_to_assets", "Asset quality", "other_real_estate_owned / total_assets", "total_assets > 0"),
    ("return_on_average_assets", "Earnings", "net_income_ttm / average_total_assets_5q_strict", "five-quarter average assets > 0"),
    ("net_interest_margin", "Earnings", "net_interest_income_ttm / average_total_assets_5q_strict", "Locked V1 label; denominator is average total assets"),
    ("interest_expense_to_average_assets", "Earnings", "total_interest_expense_ttm / average_total_assets_5q_strict", "five-quarter average assets > 0"),
    ("noninterest_income_to_average_assets", "Earnings", "total_noninterest_income_ttm / average_total_assets_5q_strict", "five-quarter average assets > 0"),
    ("efficiency_ratio", "Management proxy", "total_noninterest_expense_ttm / (net_interest_income_ttm + total_noninterest_income_ttm)", "operating revenue > 0"),
    ("loans_to_assets", "Liquidity/funding", "total_loans / total_assets", "total_assets > 0"),
    ("loans_to_deposits", "Liquidity/funding", "total_loans / total_deposits", "Candidate audit only; excluded from locked V1 features"),
    ("deposits_to_assets", "Liquidity/funding", "total_deposits / total_assets", "total_assets > 0"),
    ("securities_to_assets", "Liquidity", "(available_for_sale_securities + held_to_maturity_securities) / total_assets", "total_assets > 0"),
    ("aoci_to_assets", "Sensitivity", "signed AOCI proxy / total_assets", "total_assets > 0; signed"),
    ("annual_asset_growth", "Growth", "total_assets / exact_lag4_total_assets - 1", "five consecutive reports; lag4 > 0"),
    ("annual_loan_growth", "Growth", "total_loans / exact_lag4_total_loans - 1", "five consecutive reports; lag4 > 0"),
    ("annual_deposit_growth", "Growth", "total_deposits / exact_lag4_total_deposits - 1", "five consecutive reports; lag4 > 0"),
]
candidate_predictor_manifest = pd.DataFrame(
    predictor_definitions, columns=["predictor", "camels_area", "formula", "eligibility_or_note"]
)
candidate_predictor_manifest["locked_v1_feature"] = candidate_predictor_manifest["predictor"].isin(LOCKED_V1_BANK_PREDICTORS)
candidate_predictor_manifest["nonmissing"] = [int(panel[p].notna().sum()) for p in candidate_predictor_manifest["predictor"]]
candidate_predictor_manifest["missing"] = [int(panel[p].isna().sum()) for p in candidate_predictor_manifest["predictor"]]
candidate_predictor_manifest["negative"] = [int(panel[p].lt(0).sum()) for p in candidate_predictor_manifest["predictor"]]
candidate_predictor_manifest["minimum"] = [panel[p].min() for p in candidate_predictor_manifest["predictor"]]
candidate_predictor_manifest["maximum"] = [panel[p].max() for p in candidate_predictor_manifest["predictor"]]

predictor_numeric = panel[candidate_predictors].to_numpy(dtype="float64")
infinite_predictor_cells = int(np.isinf(predictor_numeric).sum())
predictor_gate = pd.DataFrame({
    "check": [
        "All 18 locked V1 predictors constructed", "History indicator constructed",
        "Candidate-only loans-to-deposits constructed", "No infinite candidate values",
        "AOCI signed proxy retained without transformation", "No lag1 predictor names created",
    ],
    "passed": [
        all(p in panel for p in LOCKED_V1_BANK_PREDICTORS), HISTORY_INDICATOR in panel,
        "loans_to_deposits" in panel, infinite_predictor_cells == 0,
        panel["unrealised_securities_gains_losses_proxy"].equals(panel["accumulated_other_comprehensive_income"].astype("float64")),
        not any(column.endswith("_lag1") for column in panel.columns),
    ],
})
predictor_gate["status"] = predictor_gate["passed"].map({True: "PASS", False: "FAIL"})
display(predictor_gate)
display(candidate_predictor_manifest)
if not predictor_gate["passed"].all():
    raise AssertionError("Candidate predictor construction gate failed.")


## 6. Flag reporting jumps and implausible values without altering them


In [ ]:
# Flag reporting jumps and implausible values without altering them

previous_assets = panel.groupby("idrssd", sort=False)["total_assets"].shift(1)
exact_prev_for_jump = exact_previous_calendar_quarter & previous_assets.gt(0)
panel["quarterly_asset_growth_audit"] = safe_ratio(panel["total_assets"], previous_assets) - 1.0
panel.loc[~exact_prev_for_jump, "quarterly_asset_growth_audit"] = np.nan
panel["possible_merger_related_jump"] = (
    exact_prev_for_jump & panel["quarterly_asset_growth_audit"].abs().ge(0.50)
)

merger_jump_audit = panel.loc[
    panel["possible_merger_related_jump"],
    ["report_date", "idrssd", "cert", "bank_name", "total_assets", "quarterly_asset_growth_audit"]
].copy().sort_values("quarterly_asset_growth_audit", key=lambda s: s.abs(), ascending=False)
merger_jump_audit["audit_interpretation"] = (
    "Possible merger, acquisition, charter event, restatement or genuine rapid balance-sheet change; retained unchanged"
)

implausible_rules = {
    "total_assets_below_1m": panel["total_assets"].lt(1000),
    "equity_to_assets_outside_minus1_plus1": panel["equity_to_assets"].lt(-1) | panel["equity_to_assets"].gt(1),
    "noncurrent_loans_ratio_outside_0_1": panel["noncurrent_loans_to_loans"].lt(0) | panel["noncurrent_loans_to_loans"].gt(1),
    "absolute_allowance_to_loans_above_1": panel["allowance_to_loans"].abs().gt(1),
    "loans_to_assets_outside_0_1p25": panel["loans_to_assets"].lt(0) | panel["loans_to_assets"].gt(1.25),
    "deposits_to_assets_outside_0_1p25": panel["deposits_to_assets"].lt(0) | panel["deposits_to_assets"].gt(1.25),
    "securities_to_assets_outside_0_1p25": panel["securities_to_assets"].lt(0) | panel["securities_to_assets"].gt(1.25),
    "absolute_annual_asset_growth_above_5": panel["annual_asset_growth"].abs().gt(5),
    "absolute_annual_loan_growth_above_5": panel["annual_loan_growth"].abs().gt(5),
    "absolute_annual_deposit_growth_above_5": panel["annual_deposit_growth"].abs().gt(5),
    "absolute_efficiency_ratio_above_100": panel["efficiency_ratio"].abs().gt(100),
    "negative_ttm_interest_expense": panel["total_interest_expense_ttm"].lt(0),
    "negative_trading_assets": panel["trading_assets"].lt(0),
}
value_column_for_rule = {
    "total_assets_below_1m": "total_assets",
    "equity_to_assets_outside_minus1_plus1": "equity_to_assets",
    "noncurrent_loans_ratio_outside_0_1": "noncurrent_loans_to_loans",
    "absolute_allowance_to_loans_above_1": "allowance_to_loans",
    "loans_to_assets_outside_0_1p25": "loans_to_assets",
    "deposits_to_assets_outside_0_1p25": "deposits_to_assets",
    "securities_to_assets_outside_0_1p25": "securities_to_assets",
    "absolute_annual_asset_growth_above_5": "annual_asset_growth",
    "absolute_annual_loan_growth_above_5": "annual_loan_growth",
    "absolute_annual_deposit_growth_above_5": "annual_deposit_growth",
    "absolute_efficiency_ratio_above_100": "efficiency_ratio",
    "negative_ttm_interest_expense": "total_interest_expense_ttm",
    "negative_trading_assets": "trading_assets",
}

implausible_summary_records, implausible_detail_frames = [], []
for rule, mask in implausible_rules.items():
    mask = mask.fillna(False)
    value_column = value_column_for_rule[rule]
    implausible_summary_records.append({
        "audit_rule": rule, "value_column": value_column,
        "flagged_observations": int(mask.sum()), "flagged_banks": int(panel.loc[mask, "idrssd"].nunique()),
        "minimum_flagged_value": panel.loc[mask, value_column].min(),
        "maximum_flagged_value": panel.loc[mask, value_column].max(),
        "treatment": "Retained unchanged; no global winsorisation",
    })
    if mask.any():
        detail = panel.loc[mask, ["report_date", "idrssd", "cert", "bank_name", value_column]].copy()
        detail = detail.rename(columns={value_column: "flagged_value"})
        detail["audit_rule"] = rule
        detail["value_column"] = value_column
        implausible_detail_frames.append(detail)

implausible_value_summary = pd.DataFrame(implausible_summary_records)
implausible_value_rows = (
    pd.concat(implausible_detail_frames, ignore_index=True)
    if implausible_detail_frames else
    pd.DataFrame(columns=["report_date", "idrssd", "cert", "bank_name", "flagged_value", "audit_rule", "value_column"])
)
display(implausible_value_summary)
print(f"Possible merger-related jumps: {len(merger_jump_audit):,}")
print(f"Implausible-value audit flags: {len(implausible_value_rows):,}")


## 7. Preserve uninsured-deposit fields for supplementary recent-period analysis


In [ ]:
# Preserve uninsured-deposit fields for supplementary recent-period analysis

uninsured_input_columns = [column for column in panel.columns if "uninsured" in column.lower()]
supplementary_uninsured_columns = ["report_date", "idrssd", "cert"] + uninsured_input_columns
uninsured_deposit_supplementary = panel[supplementary_uninsured_columns].copy()

if uninsured_input_columns:
    uninsured_availability_records = []
    for column in uninsured_input_columns:
        available_dates = panel.loc[panel[column].notna(), "report_date"]
        uninsured_availability_records.append({
            "source_column": column, "nonmissing": int(panel[column].notna().sum()),
            "missing": int(panel[column].isna().sum()),
            "first_available_quarter": available_dates.min() if not available_dates.empty else pd.NaT,
            "last_available_quarter": available_dates.max() if not available_dates.empty else pd.NaT,
            "analysis_role": "Supplementary recent-period only; never treated as structurally complete",
        })
    uninsured_deposit_availability = pd.DataFrame(uninsured_availability_records)
else:
    uninsured_deposit_availability = pd.DataFrame([{
        "source_column": pd.NA, "nonmissing": 0, "missing": len(panel),
        "first_available_quarter": pd.NaT, "last_available_quarter": pd.NaT,
        "analysis_role": "No uninsured-deposit source field was present in the validated Notebook 04 handoff",
    }])

display(uninsured_deposit_availability)
print(f"Uninsured-deposit source fields retained: {len(uninsured_input_columns):,}")


## 8. Create and validate the un-imputed contemporaneous source table


In [ ]:
# Create and validate the un-imputed contemporaneous source table

source_columns = ["report_date", "idrssd", "cert"] + LOCKED_V1_BANK_FEATURES
bank_predictor_source = panel[source_columns].copy().rename(columns={"report_date": "source_report_date"})
bank_predictor_source = bank_predictor_source.sort_values(
    ["source_report_date", "idrssd"], kind="mergesort"
).reset_index(drop=True)

duplicate_source_keys = bank_predictor_source.loc[
    bank_predictor_source.duplicated(["idrssd", "source_report_date"], keep=False)
]
predictor_frame = bank_predictor_source[LOCKED_V1_BANK_FEATURES]
infinite_source_cells = int(np.isinf(predictor_frame.to_numpy(dtype="float64")).sum())
source_predictor_availability = pd.DataFrame({
    "feature": LOCKED_V1_BANK_FEATURES,
    "nonmissing": [int(predictor_frame[column].notna().sum()) for column in LOCKED_V1_BANK_FEATURES],
    "missing": [int(predictor_frame[column].isna().sum()) for column in LOCKED_V1_BANK_FEATURES],
    "availability_percent": [100 * predictor_frame[column].notna().mean() for column in LOCKED_V1_BANK_FEATURES],
    "minimum": [predictor_frame[column].min() for column in LOCKED_V1_BANK_FEATURES],
    "maximum": [predictor_frame[column].max() for column in LOCKED_V1_BANK_FEATURES],
})

earliest_source = bank_predictor_source["source_report_date"].min()
first_implied_reference = earliest_source + pd.offsets.QuarterEnd(1)
expected_output_columns = ["source_report_date", "idrssd", "cert"] + LOCKED_V1_BANK_FEATURES

final_gate = pd.DataFrame({
    "check": [
        "All input observations retained", "Output contains exactly three audit identifiers plus 19 bank features",
        "Output columns match the locked manifest", "IDRSSD-source quarter is unique",
        "CERT is retained for audit", "No infinite source predictor cells",
        "No lag1 field exists", "No target-derived field exists", "No candidate-only field entered the locked source table",
        "Earliest source quarter is 2001Q1", "First implied V2 reference quarter is 2001Q2",
        "History indicator is binary", "No global winsorisation fields exist",
    ],
    "passed": [
        len(bank_predictor_source) == len(panel), bank_predictor_source.shape[1] == 22,
        list(bank_predictor_source.columns) == expected_output_columns, duplicate_source_keys.empty,
        "cert" in bank_predictor_source, infinite_source_cells == 0,
        not any(column.endswith("_lag1") for column in bank_predictor_source),
        not any(column in bank_predictor_source for column in ["failed_1yr", "failure_date", "days_to_failure"]),
        "loans_to_deposits" not in bank_predictor_source,
        earliest_source == pd.Timestamp("2001-03-31"), first_implied_reference == pd.Timestamp("2001-06-30"),
        set(bank_predictor_source[HISTORY_INDICATOR].dropna().unique()).issubset({0, 1}),
        not any("winsor" in column.lower() for column in bank_predictor_source),
    ],
})
final_gate["status"] = final_gate["passed"].map({True: "PASS", False: "FAIL"})
display(final_gate)
display(source_predictor_availability)
if not final_gate["passed"].all():
    raise AssertionError("Final contemporaneous source-table gate failed.")


## 9. Save outputs and print the final audit result


In [ ]:
# Save outputs and print the final audit result

output_tables = {
    "input_validation_gate_v2.csv": input_gate,
    "locked_v1_bank_feature_manifest_v2.csv": locked_feature_manifest,
    "source_quarter_contract_v2.csv": source_contract,
    "quarterly_flow_audit_v2.csv": quarterly_flow_audit,
    "quarterly_flow_gate_v2.csv": flow_gate,
    "ttm_flow_audit_v2.csv": ttm_flow_audit,
    "average_denominator_audit_v2.csv": average_denominator_audit,
    "history_and_trailing_measure_gate_v2.csv": history_gate,
    "candidate_predictor_manifest_v2.csv": candidate_predictor_manifest,
    "candidate_predictor_gate_v2.csv": predictor_gate,
    "merger_jump_audit_v2.csv": merger_jump_audit,
    "implausible_value_summary_v2.csv": implausible_value_summary,
    "implausible_value_rows_v2.csv": implausible_value_rows,
    "uninsured_deposit_availability_v2.csv": uninsured_deposit_availability,
    "source_predictor_availability_v2.csv": source_predictor_availability,
    "duplicate_source_keys_v2.csv": duplicate_source_keys,
    "final_validation_gate_v2.csv": final_gate,
}
for filename, table in output_tables.items():
    table.to_csv(NB05_OUTPUT_DIR / filename, index=False)

if uninsured_input_columns:
    uninsured_path = V2_PROCESSED_DATA_DIR / "uninsured_deposit_supplementary_v2.parquet"
    uninsured_deposit_supplementary.to_parquet(uninsured_path, index=False)
else:
    uninsured_path = None

temp_output = OUTPUT_PATH.with_suffix(".tmp.parquet")
bank_predictor_source.to_parquet(temp_output, index=False)
temp_output.replace(OUTPUT_PATH)
saved_source = pd.read_parquet(OUTPUT_PATH)
saved_exactly = (
    saved_source.shape == bank_predictor_source.shape
    and list(saved_source.columns) == list(bank_predictor_source.columns)
    and pd.util.hash_pandas_object(saved_source, index=True).sum()
    == pd.util.hash_pandas_object(bank_predictor_source, index=True).sum()
)
if not saved_exactly:
    raise AssertionError("Saved source table did not reload exactly.")

warning_messages = []
missing_q2_q4_flows = int(quarterly_flow_audit["q2_q4_missing_no_exact_predecessor_or_value"].sum())
if missing_q2_q4_flows:
    warning_messages.append(f"{missing_q2_q4_flows} Q2-Q4 flow cells are missing because an exact predecessor/value was unavailable")
if len(merger_jump_audit):
    warning_messages.append(f"{len(merger_jump_audit)} possible merger-related asset jumps flagged and retained")
if len(implausible_value_rows):
    warning_messages.append(f"{len(implausible_value_rows)} implausible-value audit flags recorded and retained")
if not uninsured_input_columns:
    warning_messages.append("0 uninsured-deposit source fields were available in the validated Notebook 04 handoff")

final_result = {
    "overall_status": "PASS",
    "input_observations": int(len(panel)),
    "source_table_observations": int(len(bank_predictor_source)),
    "unique_idrssd": int(bank_predictor_source["idrssd"].nunique()),
    "unique_cert": int(bank_predictor_source["cert"].nunique()),
    "reporting_quarters": int(bank_predictor_source["source_report_date"].nunique()),
    "source_period_start": str(bank_predictor_source["source_report_date"].min().date()),
    "source_period_end": str(bank_predictor_source["source_report_date"].max().date()),
    "quarterly_flow_fields": len(quarterly_flow_columns),
    "ttm_flow_fields": len(ttm_flow_columns),
    "locked_v1_bank_predictors": len(LOCKED_V1_BANK_PREDICTORS),
    "locked_bank_features_including_history": len(LOCKED_V1_BANK_FEATURES),
    "insufficient_history_rows": int(bank_predictor_source[HISTORY_INDICATOR].sum()),
    "duplicate_source_keys": int(len(duplicate_source_keys)),
    "infinite_source_predictor_cells": infinite_source_cells,
    "possible_merger_jump_flags": int(len(merger_jump_audit)),
    "implausible_value_flags": int(len(implausible_value_rows)),
    "uninsured_deposit_source_fields": len(uninsured_input_columns),
    "output_path": str(OUTPUT_PATH),
    "audit_folder": str(NB05_OUTPUT_DIR),
    "warnings": warning_messages,
}
(NB05_OUTPUT_DIR / "audit_report_v2.json").write_text(json.dumps(final_result, indent=2, default=str), encoding="utf-8")

print("=" * 78)
print("NOTEBOOK 05 V2 — FINAL QUARTERLY-FLOW AND PREDICTOR RESULT")
print("=" * 78)
print("Overall status: PASS")
print(f"Input observations: {len(panel):,}")
print(f"Source-table observations: {len(bank_predictor_source):,}")
print(f"Unique IDRSSD: {bank_predictor_source['idrssd'].nunique():,}")
print(f"Unique CERT: {bank_predictor_source['cert'].nunique():,}")
print(f"Reporting quarters: {bank_predictor_source['source_report_date'].nunique():,}")
print(f"Source period: {bank_predictor_source['source_report_date'].min().date()} to {bank_predictor_source['source_report_date'].max().date()}")
print(f"Quarterly flow fields: {len(quarterly_flow_columns):,}")
print(f"TTM flow fields: {len(ttm_flow_columns):,}")
print(f"Locked Version 1 bank predictors: {len(LOCKED_V1_BANK_PREDICTORS):,}")
print(f"Locked bank features including insufficient_history_5q: {len(LOCKED_V1_BANK_FEATURES):,}")
print(f"Rows flagged insufficient_history_5q: {bank_predictor_source[HISTORY_INDICATOR].sum():,}")
print(f"Duplicate IDRSSD-source quarter rows: {len(duplicate_source_keys):,}")
print(f"Infinite source predictor cells: {infinite_source_cells:,}")
print(f"Possible merger-related jumps: {len(merger_jump_audit):,}")
print(f"Implausible-value audit flags: {len(implausible_value_rows):,}")
print(f"Uninsured-deposit source fields: {len(uninsured_input_columns):,}")
print(f"Earliest source quarter: {earliest_source.date()}")
print(f"First supported V2 reference quarter: {first_implied_reference.date()}")
print("Imputation performed: 0")
print("Global winsorisation performed: 0")
print("Lag1 fields created: 0")
print(f"Source table: {OUTPUT_PATH}")
print(f"Audit folder: {NB05_OUTPUT_DIR}")
if warning_messages:
    print("\nNotebook 05 completed with warnings:")
    for warning in warning_messages:
        print(f"- {warning}")
    print("Review the final results and relevant warning audit summaries before Notebook 06.")
else:
    print("\nNotebook 05 completed without warnings.")
print("Do not run Notebook 06 until this predictor-source audit has been reviewed.")


In [ ]:
# Review predictor audit outputs

from pathlib import Path
import pandas as pd
from IPython.display import display

AUDIT_DIR = Path(
    "/content/drive/MyDrive/Data Final/Version 2 Main Analysis/"
    "Notebook Outputs/05_Quarterly_Flows_and_Candidate_Predictors"
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

# 1. All seven quarterly-flow audit rows
quarterly_flow_audit = pd.read_csv(
    AUDIT_DIR / "quarterly_flow_audit_v2.csv"
)

print("=" * 80)
print("1. QUARTERLY FLOW AUDIT — ALL ROWS")
print("=" * 80)
display(quarterly_flow_audit)


# 2. All implausible-value summary rows
implausible_summary = pd.read_csv(
    AUDIT_DIR / "implausible_value_summary_v2.csv"
)

print("=" * 80)
print("2. IMPLAUSIBLE-VALUE SUMMARY — ALL ROWS")
print("=" * 80)
display(implausible_summary)


# 3. Twenty largest possible merger-related asset jumps
merger_jumps = pd.read_csv(
    AUDIT_DIR / "merger_jump_audit_v2.csv",
    parse_dates=["report_date"]
)

merger_jumps["absolute_asset_growth"] = (
    merger_jumps["quarterly_asset_growth_audit"].abs()
)

largest_merger_jumps = (
    merger_jumps
    .sort_values("absolute_asset_growth", ascending=False)
    .head(20)
)

print("=" * 80)
print("3. POSSIBLE MERGER-RELATED JUMPS — 20 LARGEST")
print("=" * 80)
display(largest_merger_jumps)


# 4. Uninsured-deposit availability
uninsured_availability = pd.read_csv(
    AUDIT_DIR / "uninsured_deposit_availability_v2.csv"
)

print("=" * 80)
print("4. UNINSURED-DEPOSIT AVAILABILITY")
print("=" * 80)
display(uninsured_availability)

## Review before Notebook 06

Check the flow, unusual-value and source-data audits. Notebook 09 applies the previous-quarter match to `bank_predictor_source_v2.parquet`.
